# 03 — Fine-tune DistilBERT (Hugging Face + PyTorch)

Cách làm **thực tế** trong công việc: dùng model đã tiền huấn luyện, chỉ fine-tune 2–3 epoch với learning rate rất nhỏ.

> **Cần GPU** (Colab T4 là đủ). Trên CPU sẽ rất chậm — khi đó hãy đặt `MAX_TRAIN` nhỏ.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

MAX_TRAIN = None   # None = dùng đủ 120.000 mẫu train. Đặt vd. 5000 để chạy thử nhanh.

import torch
from data import load_ag_news
import bert_finetune as bf
import attention_viz as av
import config as C

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ — sẽ rất chậm')
print('Model:', C.BERT_NAME, '| lr =', C.BERT_LR, '| epochs =', C.BERT_EPOCHS, '| max_length =', C.BERT_MAX_LEN)
data = load_ag_news(MAX_TRAIN)

## Bước 7 — Fine-tune DistilBERT
- `lr = 3e-5` (khoảng 2e-5 – 5e-5), 3 epoch, warmup 6% + giảm tuyến tính, `max_length = 96` (không dùng 512).
- Giữ checkpoint có val accuracy tốt nhất; model được lưu vào `models/distilbert_agnews/`.

In [ ]:
kq_bert = bf.fine_tune(data)

## Thí nghiệm (tuỳ chọn): vì sao lr phải RẤT NHỎ?
Fine-tune cùng một lượng dữ liệu nhỏ (8.000 mẫu, 1 epoch) với `lr = 3e-5` và `lr = 1e-3`. Với lr lớn, trọng số tiền huấn luyện bị phá huỷ.

In [ ]:
RUN_LR_DEMO = True
if RUN_LR_DEMO:
    bf.thi_nghiem_lr(data)

## Bước 10 (tiếp) — Attention của DistilBERT
Lớp đầu và lớp cuối, trung bình các head, cho đúng 3 tin đã dùng ở notebook 02.

In [ ]:
mau = av.ve_attention_bert(data)